# 게이트 비교 재학습 — ConvNeXt-Tiny, 엄격 게이트(A) vs 관대 게이트 + 마스킹(B)

**왜 하나:** 로컬 분석(학습 없음)에서 게이트 차이는 결국 "출력물이 모델 크롭의 몇 %부터 판정할까"로 정리됐다.
- 2단계 게이트(엄격)는 사실상 출력물 약 60% 이상만 통과시킨다. 판정 비율 66%. 같은 시점 유형끼리 보면 결함 때문에 보류하지는 않는다.
- 질감 v2(관대)는 10% 이상이면 통과시킨다. 판정 비율 86%. 부분 시점 사진은 마스킹으로 베드를 가린다.
- 기존 모델을 다시 채점하면 엄격 게이트가 오탐을 거의 없애지만, 이건 재학습 없는 비교다. 같은 조건으로 **다시 학습해서** 비교해야 공정하다.

| 실험 | 옵션 | 판정 비율 |
|---|---|---|
| **A 엄격** | `--keep-list keep_hh_crop.csv` (2단계 게이트, 모델 크롭 영역에서 미리 계산), 마스킹 없음 | 66% (2,652장) |
| **B 관대** | `--gate texture --min-filament 0.1 --mask-bed --bed-erase 0.5` (질감 v2 + 마스킹 + 회색 띠 증강) | 86% (3,449장) |

나머지(고정 크롭, 512, finetune 20 epoch, 시간 블록 5-fold)는 지금까지와 같다. 회색 띠 증강은 지난번 마스킹 재학습에서 생긴 "회색 없으면 정상" 지름길(fold 4 미탐 55장)을 막기 위한 것이다.

- 순서: 시드 0의 A → B, 그다음 시드 1, 2. fold는 4 → 0 → 3 → 1 → 2(문제 구간 먼저).
- 시간: A 시드당 약 50분, B 약 75분. 시드 0 한 쌍이 약 2시간, 시드 3개 전체가 약 6시간이다.
- 시드가 끝날 때마다 결과가 저장되고, fold가 끝날 때마다 중간 결과도 저장된다.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 이미 있음
- `pipeline_train_bundle_20261004_gatecmp.zip`: **새로 올릴 것**(학습 스크립트 `--keep-list` 추가본, 게이트 v2, 엄격 게이트 통과 목록, 분할표)

노트북은 **파일 > 노트북 업로드**로 연다. 4번 셀을 누르고 **런타임 > 이후 셀 실행**을 하면 학습 → 결과 → CSV 묶기 → 런타임 반납까지 한 번에 한다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/gate_compare_20261004"
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "pipeline_train_bundle_20261004_gatecmp.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
`--keep-list`가 보이는 줄, `v2-mask`가 보이는 줄, `keep_hh_crop.csv`의 줄 수 2653(제목 1줄 + 2,652장), 마지막 줄에 4000이 나와야 한다.

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/pipeline_train_bundle_20261004_gatecmp.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/pipeline_train_bundle_20261004_gatecmp.zip -d /content/code
!grep -m1 -o -- "--keep-list" /content/code/train_pilot_classifier.py
!grep -m1 VERSION /content/code/filament_gate.py
!wc -l < /content/code/keep_hh_crop.csv
!find /content/orig -name "*.jpg" | wc -l

## 4. 본 실행 (시드 0 → 1 → 2, 시드마다 A → B)
- A 시작 몇 분 뒤 `keep-list keep_hh_crop.csv: held 1348 images {'normal': 649, 'underfill_50': 699}`가 나와야 한다.
- B 시작 몇 분 뒤 `gate texture (filament < 0.1): held 551 images {'normal': 257, 'underfill_50': 294}`가 나와야 한다.
- fold 하나가 끝날 때마다 한 줄씩 나온다. 첫 줄은 fold 4다.
- **끊겼을 때:** 이 셀을 다시 실행하면 끝난 실험은 건너뛰고, 하던 실험은 처음부터 다시 한다.
- **시간이 모자라면:** 셀을 멈추고(■) 5번부터 실행한다. 끝난 실험·fold까지 보여 준다.

In [ ]:
import shutil
BASE = ("--folds-csv /content/code/folds.csv --pilot-root /content/orig --input-column source "
        "--fixed-crop 0 250 1000 1250 --img-size 512 --modes finetune --fast --arch convnext_tiny --folds 4 0 3 1 2")
EXPS = {"A_strict": "--keep-list /content/code/keep_hh_crop.csv",
        "B_lenient_mask_erase": "--gate texture --min-filament 0.1 --mask-bed --bed-erase 0.5"}
HIDE = r"^ \|^{\|^}"     # 학습 끝의 긴 요약(JSON)을 화면에서 숨김. 같은 내용이 CSV에 있음
SEEDS = [0, 1, 2]

def run(exp, seed):
    out = f"{RESULTS}/{exp}_seed{seed}"
    if os.path.exists(f"{out}/summary.csv"):
        print("이미 끝남:", out)
        return
    shutil.rmtree(out, ignore_errors=True)   # 반만 끝난 폴더 정리
    opts = EXPS[exp]
    print(f"\n===== {exp} seed {seed}")
    !PYTHONUNBUFFERED=1 python /content/code/train_pilot_classifier.py {BASE} {opts} --seeds {seed} --output "{out}" | grep -v '{HIDE}'

for SEED in SEEDS:
    for EXP in EXPS:
        run(EXP, SEED)

## 5. 결과
- **각자 사진 집합**: 실험마다 자기 게이트를 통과한 사진 전체 기준. `n`이 판정한 장수(= 판정 비율)다.
- **공통 사진 집합**: A와 B를 둘 다 통과한 사진만으로 같은 시드끼리 비교. 게이트가 아니라 학습 방식 차이를 보는 표다.
- `vote_acc`: 최근 5장 다수결 정확도.
- 참고(지난번, 시드 0): B에서 회색 띠만 뺀 것(질감 v2 + 마스킹)은 3,449장 기준 정확도 0.972, 오탐 27, 미탐 69, fold별 .975 1.000 1.000 .961 .919.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd, json, glob
rows, folds = [], []
for exp in EXPS:
    for seed in SEEDS:
        out = f"{RESULTS}/{exp}_seed{seed}"
        for p in sorted(glob.glob(f"{out}/runs/finetune/seed{seed}/fold*/metrics.json")):
            m = json.load(open(p))
            folds.append({"exp": exp, "seed": seed, "fold": m["fold"], "acc": m["val_accuracy"], "FP": m["val_fp"], "FN": m["val_fn"]})
        if os.path.exists(f"{out}/summary.csv"):
            s = pd.read_csv(f"{out}/summary.csv", encoding="utf-8-sig").iloc[0]
            v = pd.read_csv(f"{out}/vote_metrics.csv", encoding="utf-8-sig").iloc[0]
            n = int(s[["pooled_tn_sum_over_seeds", "pooled_fp_sum_over_seeds", "pooled_fn_sum_over_seeds", "pooled_tp_sum_over_seeds"]].sum())
            rows.append({"exp": exp, "seed": seed, "n": n, "acc": s.pooled_mean_accuracy, "auroc": s.pooled_mean_auroc,
                         "FP": int(s.pooled_fp_sum_over_seeds), "FN": int(s.pooled_fn_sum_over_seeds), "vote_acc": v.vote_accuracy})
if folds:
    fd = pd.DataFrame(folds)
    print("[fold별 정확도]\n" + fd.pivot_table(index=["exp", "seed"], columns="fold", values="acc").round(3).to_string())
if rows:
    sm = pd.DataFrame(rows)
    print("\n[각자 사진 집합, 5 fold 합산]\n" + sm.round(4).to_string(index=False))
    g = sm.drop(columns="seed").groupby("exp"); avg = g.mean().round(4); avg.insert(0, "n_seeds", g.size())
    print("\n[시드 평균]\n" + avg.to_string())
    # common frames: passed by both gates, same seed
    print("\n[공통 사진 집합 (A와 B 모두 통과), 같은 시드끼리]")
    for seed in SEEDS:
        pa, pb = [f"{RESULTS}/{e}_seed{seed}/oof_predictions.csv" for e in EXPS]
        if os.path.exists(pa) and os.path.exists(pb):
            a, b = pd.read_csv(pa, encoding="utf-8-sig"), pd.read_csv(pb, encoding="utf-8-sig")
            common = set(a.source) & set(b.source)
            for name, d in (("A_strict", a), ("B_lenient_mask_erase", b)):
                d = d[d.source.isin(common)]
                fp = int(((d.label_id == 0) & (d.pred == 1)).sum()); fn = int(((d.label_id == 1) & (d.pred == 0)).sum())
                print(f"  seed {seed} {name:22s} n={len(d)} acc={1 - (fp + fn) / len(d):.4f} FP={fp} FN={fn}")

## 5-1. 덜 찍힌 사진을 만나면? (A 모델로 확인)
A(엄격)는 출력물이 덜 찍힌 사진을 학습하지도, 평가하지도 않았다. 판정할 때도 같은 게이트를 쓰면 이 사진들은 보류되지만, "보류가 길어지면 덜 찍힌 사진도 판정한다" 같은 혼합 방식을 쓰면 이 모델이 처음 보는 사진을 판정하게 된다. 그래서 A의 fold별 모델에 A가 보류했던 사진을 일부러 넣어 본다. 각 사진은 자기 구간을 학습에 안 쓴 fold 모델로 판정한다. 같은 사진에 대한 B의 판정과도 비교한다(B는 이 사진들을 마스킹해서 학습·판정했다). 시드당 1~2분 걸린다.

`coverage`는 크롭에서 출력물이 차지하는 비율(질감 v2 기준)이다.

In [ ]:
import sys, numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from torchvision.transforms.v2 import functional as TF
sys.path.insert(0, "/content/code")
from pipeline import build_model, preprocess, MEAN, STD   # same preprocessing as training (crop -> 512)
import filament_gate as fg                                 # only to measure how much of the crop the print covers

folds = pd.read_csv("/content/code/folds.csv", encoding="utf-8-sig")
val_fold = folds[folds.role == "val"].set_index("source").fold.to_dict()   # each frame is evaluated in exactly one fold
dev = torch.device("cuda")
rows = []
for seed in SEEDS:
    base = f"{RESULTS}/A_strict_seed{seed}"
    if not os.path.exists(f"{base}/summary.csv"):
        continue
    held = pd.read_csv(f"{base}/held_frames_keep_list.csv", encoding="utf-8-sig")
    held["fold"] = held.source.map(val_fold)
    for k, grp in held.groupby("fold"):
        model = build_model("convnext_tiny", f"{base}/runs/finetune/seed{seed}/fold{k}/model_state.pt", dev)
        for r in grp.itertuples():
            path = "/content/orig/" + r.source.replace(chr(92), "/")
            x, _ = preprocess(path, 512)
            crop = np.asarray(Image.open(path).convert("RGB"))[0:1000, 250:1250]
            with torch.no_grad():
                p = F.softmax(model(TF.normalize(x.to(dev).float().unsqueeze(0) / 255.0, MEAN, STD)).float(), dim=1)[0, 1].item()
            rows.append({"seed": seed, "fold": k, "source": r.source, "label": r.label, "coverage": fg.filament_ratio(crop), "prob_underfill": p})
        del model; torch.cuda.empty_cache()
if rows:
    h = pd.DataFrame(rows)
    h.to_csv(f"{RESULTS}/A_strict_on_held_frames.csv", index=False, encoding="utf-8-sig")
    h["wrong"] = (h.prob_underfill >= 0.5).astype(int) != (h.label == "underfill_50").astype(int)
    h["coverage_bin"] = pd.cut(h.coverage, [-0.01, 0.1, 0.3, 0.6, 1.01], labels=["<0.1 베드만", "0.1~0.3", "0.3~0.6", ">=0.6"])
    t = h.groupby(["coverage_bin", "label"], observed=True).agg(n=("wrong", "size"), 오류율=("wrong", "mean"), 평균_언더필확률=("prob_underfill", "mean"))
    print("[A(엄격)로 학습한 모델에 A가 보류한 사진을 넣었을 때, 시드 합산]\n" + t.round(3).to_string())
    # B's verdicts on the same frames come from B's own out-of-fold predictions (B judged these frames, masked)
    for seed in SEEDS:
        pb = f"{RESULTS}/B_lenient_mask_erase_seed{seed}/oof_predictions.csv"
        if os.path.exists(pb):
            b = pd.read_csv(pb, encoding="utf-8-sig")
            b = b[b.source.isin(set(h[h.seed == seed].source))]
            if len(b):
                print(f"[같은 사진에 대한 B(관대+마스킹) 판정, 시드 {seed}] n={len(b)} 오류율={(b.pred != b.label_id).mean():.3f}"
                      f" (정상 오탐 {int(((b.label_id == 0) & (b.pred == 1)).sum())}, 언더필 미탐 {int(((b.label_id == 1) & (b.pred == 0)).sum())})")
else:
    print("A 실험이 아직 끝나지 않음")

## 6. 결과 CSV 묶기
가중치를 뺀 CSV만 Drive `results/gate_compare_20261004_csv.zip`으로 묶는다. 이 zip을 받아서 Claude에게 주면 연속 보류 시간 같은 공정 단위 지표까지 계산해 준다.

In [ ]:
import zipfile
ZIP = f"{DRIVE_DIR}/results/gate_compare_20261004_csv.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 7. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()